<a href="https://colab.research.google.com/github/Seven8ight/AI_CAT_2/blob/main/CAT_2.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# Qn 1
As per your assigned groups for the semester, identify the ML approach and
algorithms that are relevant to the dataset & expected output of your application.
Include this information in the shared spreadsheet of the semester groups.

### Answer
- AI Application: Classification
- Algorithm/Approach: Logistic Regression
Why:


Q2:
---



In your notebook/development environment, critically detail the data cleaning
process taken from exploration to the final dataset for modelling. As part of the
EDA, derive at least 4 observations from the dataset.



Exploration:
---



The project uses a public Kaggle Dataset chosen because it explicitly has a stress-level target and self-reported data which is consistent with the project's idea to avoid wearables.


The Cleaning Process:
---





In [4]:
import pandas as pd
import numpy as np
from google.colab import files

uploaded = files.upload()
df = pd.read_csv("academic_stress_raw.csv")
df.shape

Saving academic_stress_raw.csv to academic_stress_raw (1).csv


(140, 9)

First , we read the Excel sheet using the imported pandas library and analyse rows and columns using df.shape.

In [5]:
df.head()

,Timestamp,Your Academic Stage,Peer pressure,Academic pressure from your home,Study Environment,What coping strategy you use as a student?,"Do you have any bad habits like smoking, drinking on a daily basis?",What would you rate the academic competition in your student life,Rate your academic stress index
0,24/07/2025 22:05:39,undergraduate,4,5,Noisy,Analyze the situation and handle it with intel...,No,3,5
1,24/07/2025 22:05:52,undergraduate,3,4,Peaceful,Analyze the situation and handle it with intel...,No,3,3
2,24/07/2025 22:06:39,undergraduate,1,1,Peaceful,"Social support (friends, family)",No,2,4
3,24/07/2025 22:06:45,undergraduate,3,2,Peaceful,Analyze the situation and handle it with intel...,No,4,3
4,24/07/2025 22:08:06,undergraduate,3,3,Peaceful,Analyze the situation and handle it with intel...,No,4,5


Return first 5 rows with head()

In [6]:
df.isna().sum()

,0
Timestamp,0
Your Academic Stage,0
Peer pressure,0
Academic pressure from your home,0
Study Environment,1
What coping strategy you use as a student?,0
"Do you have any bad habits like smoking, drinking on a daily basis?",0
What would you rate the academic competition in your student life,0
Rate your academic stress index,0


All columns are zero (lack null values) except one instance on Study Environment. We drop the single row instead of imputing it with the mode which would be inserting a response the student never gave.  

In [8]:
df = df.dropna(subset=["Study Environment"]).reset_index(drop=True)
df.shape

(139, 9)

Dropped the null row and on checking shape , it has reduced from 140 to 139.

In [9]:
print("Fully identical rows:", df.duplicated().sum())
print("Identical excluding Timestamp:", df.drop(columns=["Timestamp"]).duplicated().sum())

Fully identical rows: 0
Identical excluding Timestamp: 8


Check for identical rows. Also check for identical rows excluding timestamp.

In [10]:
dup_mask = df.drop(columns=["Timestamp"]).duplicated(keep=False)
df[dup_mask].sort_values(list(df.columns[1:-1]))

,Timestamp,Your Academic Stage,Peer pressure,Academic pressure from your home,Study Environment,What coping strategy you use as a student?,"Do you have any bad habits like smoking, drinking on a daily basis?",What would you rate the academic competition in your student life,Rate your academic stress index
95,26/07/2025 07:50:38,high school,3,3,Peaceful,Analyze the situation and handle it with intel...,No,3,4
108,26/07/2025 10:38:24,high school,3,3,Peaceful,Analyze the situation and handle it with intel...,No,3,4
82,25/07/2025 16:04:38,undergraduate,3,2,Peaceful,Analyze the situation and handle it with intel...,No,3,4
134,17/08/2025 13:02:04,undergraduate,3,2,Peaceful,Analyze the situation and handle it with intel...,No,3,4
3,24/07/2025 22:06:45,undergraduate,3,2,Peaceful,Analyze the situation and handle it with intel...,No,4,3
45,24/07/2025 22:34:41,undergraduate,3,2,Peaceful,Analyze the situation and handle it with intel...,No,4,3
21,24/07/2025 22:18:44,undergraduate,3,3,Peaceful,Analyze the situation and handle it with intel...,No,4,3
22,24/07/2025 22:18:55,undergraduate,3,3,Peaceful,Analyze the situation and handle it with intel...,No,4,3
50,24/07/2025 22:42:11,undergraduate,3,3,Peaceful,Analyze the situation and handle it with intel...,No,4,3
71,25/07/2025 10:30:12,undergraduate,3,3,disrupted,Analyze the situation and handle it with intel...,No,4,4


Inspect returned rows. Check on timestamp.
Decision made to keep them based on discrepancy of timestamps hence wouldn't have been responses made in one sitting.

In [14]:
df = df.rename(columns={
    "Your Academic Stage": "Academic_Stage",
    "Peer pressure": "Peer_Pressure",
    "Academic pressure from your home": "Academic_Pressure_Home",
    "Study Environment": "Study_Environment",
    "What coping strategy you use as a student?": "Coping_Strategy",
    "Do you have any bad habits like smoking, drinking on a daily basis?": "Bad_Habits",
    "What would you rate the academic  competition in your student life": "Academic_Competition",
    "Rate your academic stress index ": "Stress_Index",
})
df.columns.tolist()


['Timestamp',
 'Academic_Stage',
 'Peer_Pressure',
 'Academic_Pressure_Home',
 'Study_Environment',
 'Coping_Strategy',
 'Bad_Habits',
 'Academic_Competition',
 'Stress_Index']

Rename columns for consistency and ease.

In [18]:
for c in ["Academic_Stage", "Study_Environment", "Coping_Strategy", "Bad_Habits"]:
    print(c, "->", sorted(df[c].unique()))

Academic_Stage -> ['high school', 'post-graduate', 'undergraduate']
Study_Environment -> ['Disrupted', 'Noisy', 'Peaceful']
Coping_Strategy -> ['Analyze the situation and handle it with intellect', 'Emotional breakdown (crying a lot)', 'Social support (friends, family)']
Bad_Habits -> ['No', 'Yes', 'prefer not to say']


Check categorical columns for consistency.

In [16]:
df["Study_Environment"] = df["Study_Environment"].str.strip().str.capitalize()


Fix the discrepancy between 'disrupted' and 'Disrupted'.

In [20]:
df["Bad_Habits"].value_counts()


,count
Bad_Habits,
No,122
Yes,10
prefer not to say,7


Keep prefer not to say as an option.

In [22]:
df["Coping_Strategy"].value_counts()

,count
Coping_Strategy,
Analyze the situation and handle it with intellect,87
Emotional breakdown (crying a lot),31
"Social support (friends, family)",21


Checked , three consistent categories , no cleaning needed

In [23]:
df["Academic_Stage"].value_counts()

,count
Academic_Stage,
undergraduate,99
high school,29
post-graduate,11


In [24]:
df = df[df["Academic_Stage"] != "high school"].reset_index(drop=True)
df.shape

(110, 9)

The project is concerned with university students. Drop the high school academic stage which is a population the app isn't targeted for.

In [25]:
df = df[df["Academic_Stage"] != "post-graduate"].reset_index(drop=True)
df.shape

(99, 9)

On dropping the high school academic stage , an almost 90/10 split is left between undergraduate and post-graduate students. The decision is made to drop post-graduate students as they have an unsubstantial proportion in the dataset.
As a result , we avoid noisy and difficult to trust data from the post-graduate students.


In [ ]:
df = df[df["Academic_Stage"] == "undergraduate"].reset_index(drop=True)
df = df.drop(columns=["Academic_Stage"])
df.shape

Drop the entire column since it has zero variance. It therefore cannot split or separate anything.


4 Observations:
---
1. Scope driven filtering reduced rows of the dataset from 139 to 99 - That is, dropping of high school and post-graduate students who were out of the scope for the potential model users.
2. Academic_Stage was a keepable column but on filtering , it retained only undergraduate students - a column with zero variance is unimpactful in model training.
3.
